# 8.1 Merging ticker changes
*(For myself I skip this part. Renaming give to much headache, even though it's not that important for short-term signals. Also it does not affect the price itself.)*

This is optional. If you want it ticker-centric or don't want to get a stockanalysis.com subscription, you can just skip this part.

To get a list of ticker changes, We can loop through all tickers and query <code>Ticker Events</code> but this only works with non-delisted companies. And although you can infer it based on the ticker list by looking at whether the cik or figi has changed, that is very messy. Because a company can stay the same even if the ticker and cik/figi change. I actually did it, and it did found that it did not match the Polygon <code>Ticker Events</code>. Then I stumbled on [stockanalysis.com](https://stockanalysis.com/actions/changes/) where you can find all ticker changes for only 10 bucks a month. The first month is even free. You have to manually download them for each year and put them in the <code>stockanalysis/raw/ticker_changes/</code> folder.

After merging those we will save the result to <code>raw/renamings.csv</code> which will contain the columns <code>['from', 'to', 'now', 'date']</code>.

In [ ]:
from tickers import get_tickers, get_ticker_changes
from times import get_market_dates, get_market_calendar, last_trading_date_before
from datetime import datetime, date, time
from pathlib import Path
import pandas as pd
import shutil
import os
DATA_PATH = "../data/polygon/"
# END_DATE = last_trading_date_before(date(2026, 6, 4)) # till today only or else you will get "out of range" and don't waste loop cycles
# END_DATE will need to support updating everyday so that we only fetch the new adjustments in an idempotent way


In [ ]:
# This can be done once and then updates can be done with manually appending the list of ticker changes.
###
# Aggregate the csv's
all_ticker_changes = []
for file in os.listdir(DATA_PATH + "../stockanalysis/raw/ticker_changes/"):
    ticker_changes_year = pd.read_csv(DATA_PATH + "../stockanalysis/raw/ticker_changes/" + file, \
        parse_dates=True, index_col=0, usecols=["Date", "Old", "New"])
    all_ticker_changes.append(ticker_changes_year)

ticker_changes = pd.concat(all_ticker_changes)
ticker_changes.rename(columns={"Old": "from", 
                               "New": "to"}, inplace=True)
ticker_changes.index.names = ['date']
ticker_changes.sort_index(inplace=True)
ticker_changes.to_csv(DATA_PATH + "../stockanalysis/ticker_changes.csv")

In [ ]:
ticker_changes = pd.read_csv(DATA_PATH + "../stockanalysis/ticker_changes.csv")
print(len(ticker_changes))
ticker_changes[ticker_changes['from'] == "FB"]

# More problems

Somes there are special conditions, such as 'delinquent', which adds an extra letter at the end of the ticker. E.g. AAII went delinquent and then the ticker became AAIIE. However these are not real ticker changes so it is not contained in the stockanalysis database. However we can easily infer it from our own ticker list: if the dates are consecutive and an extra letter is added, we can infer the ticker change. We will save this to <code>raw/inferred_renamings.csv.</code>

In [ ]:
# ── Paths ─────────────────────────────────────────────────────────────────────
DATA_PATH = "../data/polygon/"
RAW       = DATA_PATH + "raw"
PROCESSED = DATA_PATH + "processed"

# ── Status suffix allow-list ──────────────────────────────────────────────────
# Restricting to exchange-assigned status letters eliminates false positives
# (two unrelated companies that happen to satisfy the date+prefix rule).
#   E  SEC-reporting delinquent
#   Q  bankruptcy / liquidation proceedings
#   D  new issue (when-issued / distribution)
# https://chatgpt.com/share/6a327201-6d30-83eb-977a-f0cae5c483ea - no need for other Suffixes like D or V since those have false positives and I want "Precision over Recall" - The Quant Golden Rule. 
# And besides, if a ticker gets to that stage, it's not gonnamake our scanner. Let's not waste time on these. 
# Out of 11k clean tickers, we only saw 1 D and 1 C and those went bankrupt within the year. 
KNOWN_SUFFIXES = {"E", "Q"}

# ── Load ticker list ──────────────────────────────────────────────────────────
tickers_v4 = pd.read_csv(
    DATA_PATH + "../tickers_v4.csv",
    parse_dates=["start_date", "end_date", "start_data", "end_data"],
)
tickers_v4 = tickers_v4[["ticker", "start_date", "end_date"]].copy()

# ── Build trading calendar from the dates in the file ─────────────────────────
# Every date in the file is a valid trading day, so the union of start/end
# dates gives us a complete-enough calendar for the next-day lookup.
all_dates    = pd.concat([tickers_v4["start_date"], tickers_v4["end_date"]]).dropna().unique()
trading_days = sorted(all_dates)

next_trading_day_map = {
    d: trading_days[i + 1]
    for i, d in enumerate(trading_days[:-1])
}

# ── Fast lookup: start_date → set of tickers starting on that day ─────────────
start_date_index = (
    tickers_v4
    .groupby("start_date")["ticker"]
    .apply(set)
    .to_dict()
)

# ── Detect status-suffix renamings ────────────────────────────────────────────
records = []

for row in tickers_v4.itertuples(index=False):   # itertuples: no per-row Series overhead
    old_ticker = row.ticker
    last_date  = row.end_date

    next_day = next_trading_day_map.get(last_date)
    if next_day is None:
        continue

    for new_ticker in start_date_index.get(next_day, set()):
        # Strip the candidate suffix first; check allow-list before the
        # (slightly more expensive) full string comparison.
        if len(new_ticker) != len(old_ticker) + 1:
            continue
        suffix = new_ticker[-1]
        if suffix not in KNOWN_SUFFIXES:          # O(1) set lookup — cheap gate
            continue
        if new_ticker[:-1] != old_ticker:         # exact root match
            continue

        records.append({
            "date":         next_day,
            "ticker_old":   old_ticker,
            "ticker_new":   new_ticker,
            "suffix_added": suffix,
        })

inferred_df = (
    pd.DataFrame(records, columns=["date", "ticker_old", "ticker_new", "suffix_added"])
    .sort_values("date")
    .reset_index(drop=True)
)

print(f"Found {len(inferred_df)} inferred renamings")
print(inferred_df)

if not inferred_df.empty:
    inferred_df.to_csv(PROCESSED / "inferred_renamings.csv", index=False)


There are a whopping 4347 ticker changes from 2003 to now that we have to take care of. But at least this was very easy to get.

Now we will use them to merge our data. We have to be aware that it is possible for a ticker to used multiple times, so the <code>ticker_changes.csv</code> may contain multiple of the same tickers in the 'from' and 'to' column. 

After processing the ticker changes we will create a <code>tickers_v5.csv</code> which will be our definitive ticker list. This contains a column 'tickers_old', which will containa list of (date_of_change, ticker) pairs. So if A changes to B on day 2, and B changes to C on day 5, tickers_old for D will contain [[2, A], [5, B]].

The process will be as follows:
* As long as we have ticker changes to process
    * Loop through <code>tickers_v4.csv</code>.
        * Get the next trading date after 'end_date_data'.
        * Search in <code>tickers_changes.csv</code> if there is a ticker change on this date.
        * If it does:
            * The stock data will be merged.
            * In <code>tickers_v4.csv</code> we will change "ticker" to the new ticker and add a list [date, ticker] to "tickers_old".
            * All other rows will be merged such as "start_date". For identifiers such as FIGI we will take the last available value. For the ID we will keep the original. If we do not do this, we might run into problems with identical IDs.
            * The row of the old ticker will be deleted
            * **We need to restart the loop!** If we don't the following can happen: Let's assume that a ticker was renamed from A -> B -> C -> D but that the order in which it appears in our ticker list is C, D, A, B. Using our loop, C gets merged with D. Then the loop checks D, which has no renamings. Then A gets merged with B. Then B gets merged with C, however that is incorrect! B should be merged with the new D, which contains C. Any double+ renamings have the risk of being in the 'wrong order'.
                * For this to work, the ticker list must be sorted on end_date.
            * Of course we must not forget that there can be adjustments on day 1 of the ticker change. There should be laws to prohibit this.

Note: if a ticker A goes OTC and then comes back and changes to B, then we will have two files: one of the A before OTC and the A+B after OTC named B. This is as intended.

In [42]:
tickers_v4 = get_tickers(v=4)
# QUICK BUG FIX, NEED TO REWRITE CODE TO MAKE IT CHRONOLOGICAL
market_dates = get_market_dates()
ticker_changes = get_ticker_changes()

END_DATE = market_dates[-1] # to not get "out of range" errors

# ── Augment ticker_changes with inferred status-suffix renamings ───────────────
# These (e.g. AAII → AAIIE) are not in stockanalysis.com but were inferred
# ── Augment ticker_changes with inferred status-suffix renamings ───────────────
try:
    inferred_path = PROCESSED + "inferred_renamings.csv"
    if os.path.exists(inferred_path):
        inferred = pd.read_csv(inferred_path, parse_dates=["date"])
        inferred["date"] = pd.to_datetime(inferred["date"]).dt.date

        tc_flat  = ticker_changes.reset_index()
        date_col = tc_flat.columns[0]

        inf_flat = (
            inferred[["date", "ticker_old", "ticker_new"]]
            .rename(columns={"date": date_col, "ticker_old": "from", "ticker_new": "to"})
        )

        n_before = len(ticker_changes)
        ticker_changes = (
            pd.concat([tc_flat, inf_flat], ignore_index=True)
            .drop_duplicates(subset=[date_col, "from"], keep="first")
            .set_index(date_col)
            .sort_index()
        )
        print(
            f"ticker_changes augmented: {n_before} → {len(ticker_changes)} entries "
            f"(+{len(inf_flat)} inferred suffix renames)"
        )
except Exception as e:
    print(f"[WARN] Inferred renamings augmentation skipped: {e}")
    print("[WARN] Main loop will proceed with unaugmented ticker_changes")

tickers_v4.insert(loc = 2, column = 'tickers_old', value = [{} for _ in range(len(tickers_v4))])

while True:
    tickers_v4 = tickers_v4.sort_values(by='end_data').reset_index(drop=True)

    # tickers_v4 gets smaller by 1 element every time we run this loop.
    for index_from, row_from in tickers_v4.copy().iterrows():
        # Get values
        type_from = row_from['type']
        if type_from == "INDEX":
            continue
        id_from = row_from['ID']
        ticker_from = row_from['ticker']
        start_date_from = row_from['start_date']
        end_date_from = row_from['end_date']
        start_data_from = row_from['start_data']
        end_data_from = row_from['end_data']

        if end_data_from == END_DATE:
            continue

        # ── DEBUG and safety checks ──────────────────────────────────────────────────────
        if pd.isna(end_data_from):
            print(f"  -> NaT detected! Skipping this ticker.")
            print(f"[WARN] index: {index_from}, ticker: {ticker_from}, "
                        f"end_data_from: {end_data_from} (type: {type(end_data_from)})")
            continue

        start_data_to = market_dates[market_dates.index(end_data_from) + 1]

        # Get ticker changes 
        # BUG_FIX
        # Try to find the change on the old ticker's last day first, then on the new ticker's first day. 
        # This fixes the issue where sometimes stockanalysis records change date as "first day of the new ticker" 
        # and sometimes as "last day of the old ticker". 
        # Example: ticker changes like UTX --> RTX did NOT get stitched cos of this. So we apply the following Fix. 
        change = ticker_changes[(ticker_changes.index == end_data_from) & (ticker_changes['from'] == ticker_from)]
        if change.empty:
            change = ticker_changes[(ticker_changes.index == start_data_to) & (ticker_changes['from'] == ticker_from)]
        # print(change)
        # change = ticker_changes[(ticker_changes.index == start_data_to) & (ticker_changes['from'] == ticker_from)]

        if change.empty:
            continue
        elif len(change) > 1:
            raise Exception("Duplicate!")
        ticker_to = change['to'].values[0]

        # Set values of new ticker
        row_to = tickers_v4[(tickers_v4['start_data'] == start_data_to) & (tickers_v4['ticker'] == ticker_to)]
        if row_to.empty:
            continue
        index_to = row_to.index[0]
        id_to = row_to['ID'].values[0]
        tickers_v4.loc[index_to, "tickers_old"][start_data_to.isoformat()] = ticker_from
        tickers_v4.loc[index_to, "start_date"] = start_date_from
        tickers_v4.loc[index_to, "start_data"] = start_data_from

        # --- IDEMPOTENCY CHECK 1 ---
        if not os.path.exists(DATA_PATH + f"processed/m1/{id_from}.parquet"):
            raise FileNotFoundError(f"[ERROR] {DATA_PATH + f"processed/m1/{id_from}.parquet"} is missing! Expected data for {ticker_from}.")

        # Do the actual merging
        from_ = pd.read_parquet(DATA_PATH + f"processed/m1/{id_from}.parquet")
        to = pd.read_parquet(DATA_PATH + f"processed/m1/{id_to}.parquet")

        # When a ticker changes, the adjustments carry over the the old ticker.
            # Get market close minute to calculate the adjustment factor, and adjust 'to'.
            # (Adjustment factor is the same throughout the day, so market close is arbitrarely chosen)
        calendar = get_market_calendar('datetime')
        start_data_to_dt = calendar.loc[start_data_to, 'regular_close']
        start_data_to_dt_bar = to.loc[start_data_to_dt]
        adjustment_factor = start_data_to_dt_bar['close'] / start_data_to_dt_bar['close_original']
        print(f"Adjustment factor of {adjustment_factor} applied to {id_from}")

        # inside the stitch loop, after reading 'to'
        print("start_data_to:", start_data_to)
        print("start_data_to_dt:", start_data_to_dt)
        bar_used = to.loc[start_data_to_dt]
        print("bar used for factor:")
        print(bar_used[['close', 'close_original']])
        print("ratio:", bar_used['close'] / bar_used['close_original'])

        any_other_bar = to.loc[to.index.date == start_data_to].iloc[0]
        print("another bar close/close_original:", any_other_bar['close'] / any_other_bar['close_original'])

        from_[['open', 'high', 'low', 'close']] = from_[['open', 'high', 'low', 'close']].multiply(adjustment_factor)
        from_ = round(from_, 4)

        # Because companies like to be annoying, a split/dividend can take place at the same time as a ticker change. We have to account for this. An example is TYDE -> OCTO with a 50:1 reverse split. However this is much easier than 5_process_raw_data.ipynb, because there is at most a single split. This is rare, but there are still a handful of cases.
        if os.path.isfile(DATA_PATH + f"raw/adjustments/{ticker_to}.csv"):
            adjustments = pd.read_csv(DATA_PATH + f"raw/adjustments/{ticker_to}.csv", parse_dates=True, index_col=0)
            adjustments.index = pd.to_datetime(adjustments.index).date
            adjustments = adjustments[(adjustments.index == start_data_to)]

            # SPLIT ADJUSTMENT
            split = adjustments[adjustments.type == 'SPLIT']
            if len(split) > 0:
                split_amount = split['amount'][0]

                from_[['open', 'high', 'low', 'close']] = from_[['open', 'high', 'low', 'close']].multiply(split_amount)

            # DIVIDEND ADJUSTMENT - REUN is the only case, not clear what happened there, likely a 'special dividend'
            dividend = adjustments[adjustments.type == 'DIV']
            if len(dividend) > 0:
                print(ticker_to)
                market_hours = get_market_calendar()
                market_hours = market_hours[['regular_close']]

                cum_div_date = end_data_from
                cum_div_time = market_hours.loc[cum_div_date][0]
                cum_div_datetime = datetime.combine(cum_div_date, cum_div_time)
                cum_div_datetime = (from_[from_.index <= cum_div_datetime].index).max()
                cum_div_close = from_.loc[cum_div_datetime, 'close']
                dividend_amount = dividend['amount'][0]
                    
                adjustment_factor_div = 1 - dividend_amount/cum_div_close

                from_[['open', 'high', 'low', 'close']] = from_[['open', 'high', 'low', 'close']].multiply(adjustment_factor_div)
            
            # ROUNDING
            if len(split) > 0 or len(dividend) > 0:
                from_ = round(from_, 4)
                from_['turnover'] = from_['turnover'].astype(int)

        # # If on the old ticker, there are divs/splits on start_data_to (start of new ticker), then something is wrong.
        # if os.path.isfile(DATA_PATH + f"raw/adjustments/{ticker_from}.csv"):
        #     adjustments = pd.read_csv(DATA_PATH + f"raw/adjustments/{ticker_from}.csv", parse_dates=True, index_col=0)
        #     adjustments.index = pd.to_datetime(adjustments.index).date
        #     adjustments = adjustments[(adjustments.index == start_data_to)]
        #     assert len(adjustments) == 0

        # Remove the 'from' ticker, then paste the 'from' and 'to' ticker to m1_renamed for debugging purposes.
        _ = shutil.move(DATA_PATH + f'processed/m1/{id_from}.parquet', DATA_PATH + f'processed/m1_renamed/{id_from}.parquet')
        _ = shutil.copyfile(DATA_PATH + f'processed/m1/{id_to}.parquet', DATA_PATH + f'processed/m1_renamed/{id_to}.parquet')

        pd.concat([from_, to]).to_parquet(DATA_PATH + f"processed/m1/{id_to}.parquet", engine="fastparquet", row_group_offsets=25000)

        tickers_v4.drop(index_from, inplace=True)
        tickers_v4.reset_index(inplace=True, drop=True)
        
        print(f"Ticker change {ticker_from} -> {ticker_to} on {start_data_to} has been processed")
        print(f"{index_from/len(tickers_v4)*100:.1f}% | Length of tickers_v4 is {len(tickers_v4)}")
        break

    # If we have reached the end of the loop, it means we have processed everything. Then we can stop.
    if index_from >= (len(tickers_v4)-1):
        break
    
tickers_v4 = tickers_v4.sort_values(by='ID').reset_index(drop=True)

tickers_v4.to_csv("../data/tickers_v5.csv")

Adjustment factor of 0.9949378881987577 applied to GSAH-2018-07-30
start_data_to: 2020-02-10
start_data_to_dt: 2020-02-10 15:59:00
bar used for factor:
close             12.8148
close_original      12.88
Name: 2020-02-10 15:59:00, dtype: object
ratio: 0.9949378881987577
another bar close/close_original: 0.994939393939394
Ticker change GSAH -> VRT on 2020-02-10 has been processed
0.0% | Length of tickers_v4 is 11285
Adjustment factor of 0.86445108587368 applied to UTX-2016-06-06
start_data_to: 2020-04-03
start_data_to_dt: 2020-04-03 15:59:00
bar used for factor:
close             43.3868
close_original      50.19
Name: 2020-04-03 15:59:00, dtype: object
ratio: 0.86445108587368
another bar close/close_original: 1.1046212261529913
Ticker change UTX -> RTX on 2020-04-03 has been processed
0.0% | Length of tickers_v4 is 11284
Adjustment factor of 0.19430549059956972 applied to SNE-2016-06-06
start_data_to: 2021-04-01
start_data_to_dt: 2021-04-01 15:59:00
bar used for factor:
close          

# Dev's Extremely Important Improvement

```python
# Get ticker changes 
# BUG_FIX
# Try to find the change on the old ticker's last day first, then on the new ticker's first day. 
# This fixes the issue where sometimes stockanalysis records change date as "first day of the new ticker" 
# and sometimes as "last day of the old ticker". 
# Example: ticker changes like UTX --> RTX did NOT get stitched cos of this. So we apply the following Fix. 
change = ticker_changes[(ticker_changes.index == end_data_from) & (ticker_changes['from'] == ticker_from)]
if change.empty:
    change = ticker_changes[(ticker_changes.index == start_data_to) & (ticker_changes['from'] == ticker_from)]
# print(change)
# change = ticker_changes[(ticker_changes.index == start_data_to) & (ticker_changes['from'] == ticker_from)]
```

In [39]:
tickers_v5 = get_tickers(v=5)
renamings = tickers_v5[tickers_v5["tickers_old"].str.len() > 2] # These were renamed
print(len(renamings))
tickers_v5[tickers_v5['ticker'] == 'META']

11


,ID,ticker,tickers_old,name,active,start_date,end_date,start_data,end_data,type,cik,composite_figi
6405,META-2022-06-09,META,{'2022-06-09': 'FB'},"Meta Platforms, Inc. Class A Common Stock",True,2016-06-06,2026-06-03,2016-06-06,2026-06-03,CS,1326801.0,BBG000MM2P62


Tickers that were renamed multiple times:

In [40]:
multiple_renamings = tickers_v5[tickers_v5["tickers_old"].str.len() > 23]
print(len(multiple_renamings))
multiple_renamings.head(2)

0


,ID,ticker,tickers_old,name,active,start_date,end_date,start_data,end_data,type,cik,composite_figi


Now we have 5 tickers lists. These are:
1. Basic ticker list with a lot of incorrect duplications.
2. Duplications merged and incorrect tickers removed.
3. ETFs added.
4. Data start/end dates added.
5. Renamings merged.
Only the last should be used in backtesting.

If Polygon just provided these from the start, it would have saved countless hours. But at least I learned some Python I guess. And at least Polygon does not ask thousands.

# BUG

# It seems that some of the data is not stitch-adjusted and will incorrectly show the discrepancy as a gap-up! 

# Check if other tickers have the same issue or just this one is not getting adjustment_factor applied correctly cos of maybe dates discrepancy! 

In [ ]:
# Perfect!

# Question:
# For UTX → RTX, why is the "close" of 74.2996 on 2020-04-02 19:59:00 lower than "close_original" of 85.95 on that day 
# but "close" of 59.6385 on 2020-04-03 higher than the "close_original" of  53.99 on that day!?
# Adjustment factor of 0.86445108587368 applied to UTX-2016-06-06
# Adjustment factor calculated using real trades EOD:
# 2020-04-03 15:59:00	43.1837	43.4128	43.0670	43.3868	50.190	10146811	202168.0	1192.0	True
# NOT using backfilled bars in the pre-mkt that are filled to just get price continuity somewhat rather than be used for calculations.
# Conclusion: It's all good! Adjustment factor of 0.86445108587368 is the correct adjustment factor. 

bars = pd.read_parquet(f"{DATA_PATH}processed/m1/RTX-2020-04-03.parquet")
# filtered = bars.loc['2020-04-02 19:57:00':'2020-04-03 04:02:00']
filtered = bars.loc['2020-04-02 19:57:00':'2020-04-03 15:59:00']
display(filtered)

,open,high,low,close,close_original,turnover,volume,transactions,tradeable
datetime,,,,,,,,,
2020-04-02 19:57:00,74.2996,74.2996,74.2996,74.2996,85.95,21487,250.0,3.0,True
2020-04-02 19:58:00,74.2996,74.2996,74.2996,74.2996,85.95,0,0.0,0.0,False
2020-04-02 19:59:00,74.2996,74.2996,74.2996,74.2996,85.95,0,0.0,0.0,False
2020-04-03 04:00:00,59.6385,59.6385,59.6385,59.6385,53.99,0,0.0,0.0,False
2020-04-03 04:01:00,59.6385,59.6385,59.6385,59.6385,53.99,0,0.0,0.0,False
2020-04-03 04:02:00,59.6385,59.6385,59.6385,59.6385,53.99,0,0.0,0.0,False


In [ ]:
# Perfect!
# Adjustment factor of 40.0 applied to SRNG-2021-04-19 

bars = pd.read_parquet(f"{DATA_PATH}processed/m1/DNA-2021-09-17.parquet")
filtered = bars.loc['2021-09-16 19:57:00':'2021-09-17 04:02:00']
display(filtered)

,open,high,low,close,close_original,turnover,volume,transactions,tradeable
datetime,,,,,,,,,
2021-09-16 19:57:00,467.6,468.8,467.6,468.8,11.72,15493,1322.0,8.0,True
2021-09-16 19:58:00,467.6,467.6,466.0,466.0,11.65,4205,361.0,4.0,True
2021-09-16 19:59:00,467.6,470.0,467.6,470.0,11.75,9071,772.0,2.0,True
2021-09-17 04:00:00,518.0,518.0,518.0,518.0,12.95,0,0.0,0.0,False
2021-09-17 04:01:00,518.0,518.0,518.0,518.0,12.95,0,0.0,0.0,False
2021-09-17 04:02:00,518.0,518.0,518.0,518.0,12.95,0,0.0,0.0,False


In [ ]:
# Perfect! 
# Adjustment factor of 0.19430549059956972 applied to SNE-2016-06-06

bars = pd.read_parquet(f"{DATA_PATH}processed/m1/SONY-2021-04-01.parquet")
filtered = bars.loc['2021-03-31 19:57:00':'2021-04-01 04:02:00']
display(filtered)

,open,high,low,close,close_original,turnover,volume,transactions,tradeable
datetime,,,,,,,,,
2021-03-31 19:57:00,20.5983,20.5983,20.5983,20.5983,106.010,0,0.0,0.0,False
2021-03-31 19:58:00,20.5983,20.5983,20.5983,20.5983,106.010,0,0.0,0.0,False
2021-03-31 19:59:00,20.5983,20.5983,20.5983,20.5983,106.010,0,0.0,0.0,False
2021-04-01 04:00:00,20.7052,20.7052,20.7052,20.7052,106.125,0,0.0,0.0,False
2021-04-01 04:01:00,20.7052,20.7052,20.7052,20.7052,106.125,0,0.0,0.0,False
2021-04-01 04:02:00,20.7052,20.7052,20.7052,20.7052,106.125,0,0.0,0.0,False


In [ ]:
# Perfect! 
# 4% Gap up
# Adjustment factor of 1.0 applied to BITF-2021-06-21

bars = pd.read_parquet(f"{DATA_PATH}processed/m1/KEEL-2026-04-06.parquet")
filtered = bars.loc['2026-04-02 19:57:00':'2026-04-06 04:02:00']
display(filtered)

In [ ]:
# Perfect!

bars = pd.read_parquet(f"{DATA_PATH}processed/m1/META-2022-06-09.parquet")
filtered = bars.loc['2022-06-08 19:57:00':'2022-06-09 04:02:00']
display(filtered)

In [ ]:
# Perfect!

bars = pd.read_parquet(f"{DATA_PATH}processed/m1/VSXY-2026-06-02.parquet")
filtered = bars.loc['2026-06-01 19:57:00':'2026-06-02 04:02:00']
display(filtered)

In [ ]:
# Perfect!

bars = pd.read_parquet(f"{DATA_PATH}processed/m1/XYZ-2025-01-21.parquet")
filtered = bars.loc['2025-01-17 19:57:00':'2025-01-21 04:02:00']
display(filtered)

In [ ]:
# Perfect!

bars = pd.read_parquet(f"{DATA_PATH}processed/m1/VRT-2020-02-10.parquet")
filtered = bars.loc['2020-02-07 19:57:00':'2020-02-10 04:02:00']
display(filtered)

In [ ]:
# Perfect!

bars = pd.read_parquet(f"{DATA_PATH}processed/m1/MRSH-2026-01-14.parquet")
filtered = bars.loc['2026-01-13 19:57:00':'2026-01-14 04:02:00']
display(filtered)

In [ ]:
# Perfect!

bars = pd.read_parquet(f"{DATA_PATH}processed/m1/LOGC-2024-05-13.parquet")
filtered = bars.loc['2024-05-10 19:57:00':'2024-05-13 04:02:00']
display(filtered)

In [ ]:
# Perfect!

bars = pd.read_parquet(f"{DATA_PATH}processed/m1/BKKT-2021-10-18.parquet")
filtered = bars.loc['2021-10-15 19:57:00':'2021-10-18 04:02:00']
display(filtered)

In [ ]:
print(f"Last date in market_dates: {market_dates[-1]}")
print(f"end_data_from: {end_data_from}")
print(f"END_DATE: {END_DATE}")
print(f"Index of end_data_from: {market_dates.index(end_data_from)}")


In [ ]:
tickers_v5 = get_tickers(v=5)
renamings = tickers_v5[tickers_v5["tickers_old"].str.len() > 2] # These were renamed
print(len(renamings))
tickers_v5[tickers_v5['ticker'] == 'META']

In [ ]:
# Check for Negative Prices again! Think I saw one!

# List of tickers with Negative prices
PROCESSED_PATH = "../data/polygon/processed/m1"
FILE_EXT = ".parquet"

for file in os.listdir(PROCESSED_PATH):
    if file.endswith(FILE_EXT):
        ticker = file.replace(FILE_EXT, "")
        file_path = os.path.join(PROCESSED_PATH, file)        
        
        df = pd.read_parquet(file_path)
        
        if (df['close'] < 0).any():
            print(f'{ticker} has negative prices!')

# Verify that all the Data Cleaning processes have been applied correctly! Especially for stocks with ticker changes. 

## Verify OHLC and Volume. Read parquet files to verify backfills, transactions, turnover, volume, etc.
Example Ticker Changes: FB/META, BITF/KEEL, XON/XOM, WTW/WW, MMC/MRSH, LB/VSCO, RTN/RTX, GEN (formerly symantec)


## Verify many tickers by Plotting their charts and comparing against Tradingview 

Example Ticker Changes: FB/META, BITF/KEEL, XON/XOM, WTW/WW, MMC/MRSH, LB/VSCO, RTN/RTX, GEN (formerly symantec)

Furthermore, several large-cap companies have completely changed the exchange they trade on rather than their ticker—such as T-Mobile, Honeywell, and Kraft Heinz—which transferred their primary listings from the NYSE to the Nasdaq.
TMUS, HON, KHC

T-Mobile: Changed from TMUS to T (or various tracker shifts) following their merger with Sprint.
TMUS/T


In [ ]:
# Plots of Daily data

# tickers = ['META', 'KEEL', 'XOM', 'WW', 'MRSH', 'VSCO', 'RTX', 'GEN', 'HON', 'KHC', 'TMUS', 'T']
tickers = ['META', 'KEEL']


for ticker in tickers:
    all_adjusted_bars = pd.read_parquet(f"{DATA_PATH}processed/m1/{ticker}.parquet")

    # Resample to daily OHLC
    daily = all_adjusted_bars.resample('1D').agg({'open': 'first', 
                                    'high': 'max', 
                                    'low': 'min', 
                                    'close': 'last'})

    # Remove any rows where all OHLC columns are NaN (if any)
    daily.dropna(inplace=True)

    # Plot last 10,000 days
    mpf.plot(daily.tail(10000), type='candle', show_nontrading=False, warn_too_much_data=1000000)

In [ ]:
# Plots of 1-min data

# List of tickers (or just one)
tickers = ['META', 'KEEL', 'XOM', 'WW', 'MRSH', 'VSCO', 'RTX', 'GEN', 'HON', 'KHC', 'TMUS', 'T']

for ticker in tickers:
    # Read the 1-minute parquet file
    df = pd.read_parquet(f"{DATA_PATH}processed/m1/{ticker}.parquet")

    # Ensure the index is datetime (it usually is)
    # If not, set it: df.index = pd.to_datetime(df.index)

    # Drop any rows where all OHLC are NaN
    df.dropna(subset=['open', 'high', 'low', 'close'], how='all', inplace=True)

    # --- Plot the last N bars (e.g., 10000 minutes ≈ 12-13 trading days) ---
    # Adjust the number based on your screen/memory
    mpf.plot(df.tail(10000), 
             type='candle',          # or 'candle' for candlesticks
             volume=True,         # set to True if you have volume column
             show_nontrading=False,
             warn_too_much_data=1000000,
             title=f"{ticker} - 1-min OHLC",
             figsize=(12, 6))

Tickers that were renamed multiple times:

In [ ]:
multiple_renamings = tickers_v5[tickers_v5["tickers_old"].str.len() > 23]
print(len(multiple_renamings))
multiple_renamings.head(2)

# Incorrect Stitching (UTX --> RTX example) of spin-offs that are a completely new company with a completely new structure!
What actually happened with UTX → RTX
In April 2020, United Technologies (UTX) didn’t simply rename. It spun off Carrier (CARR) and Otis (OTIS) and then merged the remaining business with Raytheon to form RTX.
- Before the event, a UTX share represented ownership in all three businesses (aerospace + Carrier + Otis).
- After the event, an RTX share represented only the aerospace/defense portion. The other two became separate stocks distributed to shareholders.

The raw price drop from ~85.95 to ~53.99 is the market’s repricing after the two spin-offs were removed. It is not a split, a dividend, or an exchange‑ratio conversion—it’s a genuine change in what the stock represents. The stockanalysis.com database likely recorded this as a “ticker change” because UTX did disappear and RTX emerged, but the underlying assets are NOT the same. It's a completely new company with a completely new structure!

### Incorrect Stitching is detrimental to our "Gap", "Momentum" and Trend-following strategies! So pretty much our ENTIRE System! MUST FIX THIS!
For multi‑week swing systems that rely on price continuity (momentum, moving averages, etc.), a gap of this size is damaging. But the damage is caused by the spin‑off itself, not by the stitching algorithm. 

# Solution: Code to detect whether a gap is a ticker change gap (like a new company with completely new structure vs old company - do not stitch) or a genuine news gap (preserve)?
Break the chain – detect large discontinuities and do not stitch. The original design discussion suggested a 20% price‑gap check (using the first new open vs old last close) to flag and optionally break such cases. UTX/RTX would easily trigger that.
Add the gap‑detection filter and break the chain for such events. 

Steps:
- You can scan your entire database for large price gaps at ticker‑change boundaries with the script below. It compares the last raw close of the old ticker to the first raw open of the new ticker — exactly the gap that the stitch does not close, so it reveals structural breaks like the UTX → RTX spin‑off.
- compute `gap = (new_open / old_close) - 1`
- Flag any gap outside ±20‑25% (you can tweak the thresholds).

Expected results:
- For pure renames (FB→META, SQ→XYZ), the gap will be ~0%.
- For SPACs (SRNG→DNA) you’ll see huge gaps (40x etc.) – these are genuine structural changes.
- For the UTX→RTX case, the gap will be about -37% (53.99/85.95 - 1), and it will be flagged.

You’ll probably find a few dozen to a few hundred events out of ~5,000 changes. Those are the cases where blindly stitching creates a discontinuity that your momentum/MA signals would see.

What to do with the results:
Keep the gap list as a reference – you don’t have to break these chains automatically, but you can flag them in your ticker list (e.g., add a structural_break = True column) so that any strategy can optionally skip them.

If you want to break the chain, add a condition before the stitch loop that checks the gap and continues (skips stitching) if it exceeds your threshold. This will leave the old ticker’s history separate, and your backtest will treat the new ticker as a fresh listing.

For your OHLCV‑only swing strategies, the earlier LLM advice still stands: the cost of lost history may be higher than the noise from the gap, so you might choose to stitch everything anyway. At least now you’ll know exactly which tickers have a price discontinuity.




In [ ]:
# Code to detect huge discontinuities
# Here’s a parallelized version using concurrent.futures.ProcessPoolExecutor. 
# It will scan your entire ticker_changes.csv in a fraction of the time by distributing the work across all your CPU cores.
# On a typical modern machine with 8+ cores, this will scan 5,000 changes in under a minute.



In [63]:
import pandas as pd
import os
from datetime import datetime, time, date
import ast
from concurrent.futures import ThreadPoolExecutor, as_completed
import multiprocessing

DATA_PATH = "../data/polygon/"
PROCESSED_PATH = os.path.join(DATA_PATH, "processed", "m1")

# Load the final (stitched) ticker list
tickers_v5 = pd.read_csv("../data/tickers_v5.csv")
tickers_v5['tickers_old'] = tickers_v5['tickers_old'].apply(
    lambda x: ast.literal_eval(x) if isinstance(x, str) and x != '{}' else {}
)

# Flatten all rename events: each is (old_ticker, new_ticker, change_date, current_id)
events = []
for _, row in tickers_v5.iterrows():
    ticker_current = row['ticker']
    tid = row['ID']
    for chg_date_str, old_ticker in row['tickers_old'].items():
        change_date = date.fromisoformat(chg_date_str)
        events.append((old_ticker, ticker_current, change_date, tid))

print(f"Found {len(events)} rename events in the stitched database.")

# Threshold for flagging (now in percentage points, e.g., 20 means ±20%)
GAP_THRESHOLD = 20.0

def check_merged_file(args):
    old_ticker, new_ticker, change_date, tid = args
    parquet_path = os.path.join(PROCESSED_PATH, f"{tid}.parquet")
    if not os.path.exists(parquet_path):
        return None

    try:
        # Read only the bars around the change date to save memory
        # We'll load a range: one day before and one day after
        start_dt = datetime.combine(change_date, time(0)) - pd.Timedelta(days=1)
        end_dt   = datetime.combine(change_date, time(23,59)) + pd.Timedelta(days=1)
        df = pd.read_parquet(parquet_path,
                             filters=[("datetime", ">=", start_dt),
                                      ("datetime", "<=", end_dt)])
        if df.empty:
            return None

        # Ensure index is datetime
        if not isinstance(df.index, pd.DatetimeIndex):
            df.index = pd.to_datetime(df.index)

        # The old ticker's last bar: the last bar before change_date
        old_part = df[df.index.date < change_date]
        if old_part.empty:
            return None
        old_last = old_part.iloc[-1]
        old_last_close = old_last['close']

        # The new ticker's first tradeable bar on or after change_date
        new_part = df[(df.index.date >= change_date) & (df['tradeable'] == True)]
        if new_part.empty:
            return None
        new_first = new_part.iloc[0]
        new_first_close = new_first['close']

        # Gap after stitch: ((new_close / old_close) - 1) * 100  (percentage)
        gap_pct = ((new_first_close / old_last_close) - 1) * 100

        if abs(gap_pct) > GAP_THRESHOLD:
            return {
                'old_ticker': old_ticker,
                'new_ticker': new_ticker,
                'change_date': change_date,
                'old_last_close': round(old_last_close, 4),
                'new_first_close': round(new_first_close, 4),
                'gap_pct': round(gap_pct, 4),   # now in percent
                'flag': 'structural_gap'
            }
    except Exception:
        return None
    return None

# Run in parallel
max_workers = min(32, (multiprocessing.cpu_count() or 4) * 4)
print(f"Scanning {len(events)} events using {max_workers} threads…")

gaps = []
with ThreadPoolExecutor(max_workers=max_workers) as executor:
    futures = {executor.submit(check_merged_file, ev): ev for ev in events}
    for future in as_completed(futures):
        result = future.result()
        if result is not None:
            gaps.append(result)

gaps_df = pd.DataFrame(gaps)
print(f"Structural gaps found: {len(gaps_df)}")
if not gaps_df.empty:
    gaps_df['abs_gap'] = gaps_df['gap_pct'].abs()
    top = gaps_df.sort_values('abs_gap', ascending=False).head(10)
    print("\nTop 10 largest residual gaps (in %):")
    print(top[['old_ticker', 'new_ticker', 'change_date', 'gap_pct']])
    out_path = os.path.join(DATA_PATH, "processed", "structural_gaps_merged.csv")
    gaps_df.to_csv(out_path, index=False)
    print(f"Full list saved to {out_path}")
else:
    print("No structural gaps detected.")

Found 11 rename events in the stitched database.
Scanning 11 events using 32 threads…
Structural gaps found: 1

Top 10 largest residual gaps (in %):
  old_ticker new_ticker change_date  gap_pct
0        UTX        RTX  2020-04-03 -37.1843
Full list saved to ../data/polygon/processed/structural_gaps_merged.csv


# Data Cleaning is Finished!
Now we have multiple ticker lists. These are:
1. Basic ticker list with a lot of incorrect duplications.
2. Duplications merged and incorrect tickers removed.
3. Data start/end dates added (in 03_tickers.ipynb). 
4. ETFs added.
5. Adjustments (Splits, Dividends) applied whilst keeping `close_original` for PIT Price.
6. Data Cleaned thoroughly (with "backfills" applied to all 1-min bars - especially useful for Pre-mkt bars) under 07_process.ipynb
7. Renamings (ticker changes) merged and stitched (FB/META, BITF/KEEL). tickers_definitive_with_renamings.csv

**Only the last should be used in backtesting.**

If Polygon just provided these from the start, it would have saved countless hours. But at least I learned some Python I guess. And at least Polygon does not ask thousands.

# 8.2 Updates
Rerun the file after setting END_DATE and updating the list of renamings.